### The Model study for 0 percent oversampled- original dataset 

#### RANDOM FOREST AND LIGHT GRADIENT BOOSTING

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.metrics import classification_report, make_scorer
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier
import joblib
import os

# 1. Loading my dataset (reduced enhanced dataset with no oversampling)
df = pd.read_csv("full_data_unhealthy_imputed_reduced_enhanced.csv")
targets = ['oestrus', 'calving', 'lameness', 'mastitis']
features = ['IN_ALLEYS', 'REST', 'EAT', 'ACTIVITY_LEVEL', 'hour_bin']

# 2. Performing feature engineering to add cyclical time encoding and behavioral ratios
def create_features(X):
    X = X.copy()
    X['hour_sin'] = np.sin(2 * np.pi * X['hour_bin'] / 24)
    X['hour_cos'] = np.cos(2 * np.pi * X['hour_bin'] / 24)
    X['eat_rest_ratio'] = X['EAT'] / (X['REST'] + 1e-6)
    X['activity_rest_ratio'] = np.abs(X['ACTIVITY_LEVEL']) / (X['REST'] + 1e-6)
    return X.drop(columns=['hour_bin'])

X = create_features(df[features])
y = df[targets]

# 3. Splitting my dataset into training and testing sets (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 4. Defining the scoring metric (weighted F1-score)
from sklearn.metrics import f1_score
scorer = make_scorer(lambda yt, yp: f1_score(yt, yp, average="weighted"))

# 5. Setting up cross-validation strategy (3-fold CV)
cv = KFold(n_splits=3, shuffle=True, random_state=42)

# 6. Creating output folder to save trained models
os.makedirs("models/0pct", exist_ok=True)

# 7. Defining a function to tune hyperparameters and train a model using GridSearchCV
def tune_and_train(X_train, y_train, model_type):
    if model_type == "rf":
        base_model = RandomForestClassifier(random_state=42, n_jobs=-1)
        param_grid = {
            "n_estimators": [100, 150],
            "max_depth": [10, 20],
            "min_samples_split": [2, 5],
            "class_weight": ["balanced"]
        }
    else:  # LightGBM
        base_model = LGBMClassifier(random_state=42, n_jobs=-1)
        param_grid = {
            "n_estimators": [200, 300],
            "max_depth": [12, 16],
            "learning_rate": [0.05, 0.1],
            "subsample": [0.8],
            "colsample_bytree": [0.8],
            "class_weight": ["balanced"]
        }

    grid = GridSearchCV(
        base_model,
        param_grid,
        scoring=scorer,
        cv=cv,
        verbose=2,
        n_jobs=-1
    )
    grid.fit(X_train, y_train)
    print(f"Best params for {model_type.upper()}: {grid.best_params_}")
    return grid.best_estimator_

# 8. Training Random Forest and LightGBM models for each target and saving the best models
for model_type in ["rf", "lgbm"]:
    print(f"\n=== Training {model_type.upper()} models with 0% oversampling ===\n")
    for target in targets:
        print(f"--- {target} ---")
        best_model = tune_and_train(X_train, y_train[target], model_type)
        y_pred = best_model.predict(X_test)
        print(f"\nClassification Report for {model_type.upper()} - {target}:\n")
        print(classification_report(y_test[target], y_pred))
        # Saving the trained model
        joblib.dump(best_model, f"models/0pct/{model_type}_{target}.pkl")



=== Training RF models with 0% oversampling ===

--- oestrus ---
Fitting 3 folds for each of 8 candidates, totalling 24 fits
Best params for RF: {'class_weight': 'balanced', 'max_depth': 20, 'min_samples_split': 5, 'n_estimators': 150}

Classification Report for RF - oestrus:

              precision    recall  f1-score   support

           0       1.00      0.99      0.99     63393
           1       0.04      0.10      0.06       241

    accuracy                           0.99     63634
   macro avg       0.52      0.55      0.53     63634
weighted avg       0.99      0.99      0.99     63634

--- calving ---
Fitting 3 folds for each of 8 candidates, totalling 24 fits
Best params for RF: {'class_weight': 'balanced', 'max_depth': 20, 'min_samples_split': 2, 'n_estimators': 150}

Classification Report for RF - calving:

              precision    recall  f1-score   support

           0       1.00      1.00      1.00     63497
           1       0.09      0.12      0.10       137

 

#### MLP MODEL

In [5]:

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import f1_score, classification_report
import os

# 2. Loading my dataset (reduced enhanced dataset with no oversampling)
df = pd.read_csv("full_data_unhealthy_imputed_reduced_enhanced.csv")
targets = ['oestrus', 'calving', 'lameness', 'mastitis']
features = ['IN_ALLEYS', 'REST', 'EAT', 'ACTIVITY_LEVEL', 'hour_bin']

# 3. Performing feature engineering to add cyclical encoding of time and behavioral ratios
def create_features(X):
    X = X.copy()
    X['hour_sin'] = np.sin(2 * np.pi * X['hour_bin'] / 24)
    X['hour_cos'] = np.cos(2 * np.pi * X['hour_bin'] / 24)
    X['eat_rest_ratio'] = X['EAT'] / (X['REST'] + 1e-6)
    X['activity_rest_ratio'] = np.abs(X['ACTIVITY_LEVEL']) / (X['REST'] + 1e-6)
    return X.drop(columns=['hour_bin'])

X = create_features(df[features]).values
y = df[targets].values

# 4. Splitting my dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 5. Creating a custom PyTorch Dataset to load data efficiently
class CattleDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
    def __len__(self): 
        return len(self.X)
    def __getitem__(self, idx): 
        return self.X[idx], self.y[idx]

# 6. Defining the MLP model architecture for multi-output classification
class MLPModel(nn.Module):
    def __init__(self, input_dim, hidden1=128, hidden2=64, dropout=0.3, output_dim=4):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, hidden1),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden1, hidden2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden2, output_dim),
            nn.Sigmoid()
        )
    def forward(self, x): 
        return self.network(x)

# 7. Creating a function to train one fold of the model with given hyperparameters
def train_one_fold(X_fold_train, y_fold_train, params, epochs=20, batch_size=256):
    dataset = CattleDataset(X_fold_train, y_fold_train)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
    
    # Extract model parameters (exclude learning rate)
    model_params = {k: v for k, v in params.items() if k != "lr"}
    
    # Initialize model
    model = MLPModel(X_fold_train.shape[1], **model_params, output_dim=y_fold_train.shape[1])
    
    # Define loss and optimizer
    criterion = nn.BCELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=params["lr"])
    
    # Training loop
    model.train()
    for _ in range(epochs):
        for xb, yb in loader:
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()
    return model
# 8. Performing 3-fold cross-validation manually to tune hyperparameters for MLP
kf = KFold(n_splits=3, shuffle=True, random_state=42)
param_grid = [
    {"hidden1": 128, "hidden2": 64, "dropout": 0.3, "lr": 0.001},
    {"hidden1": 256, "hidden2": 128, "dropout": 0.4, "lr": 0.0005},
]
best_score = -1
best_params = None

for params in param_grid:
    scores = []
    for train_idx, val_idx in kf.split(X_train):
        model = train_one_fold(X_train[train_idx], y_train[train_idx], params)
        model.eval()
        with torch.no_grad():
            preds = (model(torch.tensor(X_train[val_idx], dtype=torch.float32)) > 0.5).int().numpy()
        score = f1_score(y_train[val_idx], preds, average="weighted")
        scores.append(score)
    avg_score = np.mean(scores)
    print(f"Params {params} -> CV weighted F1-score: {avg_score:.4f}")
    if avg_score > best_score:
        best_score = avg_score
        best_params = params

print("Best hyperparameters selected:", best_params)

# 9. Retraining the MLP on the full training data with the best hyperparameters
final_model = train_one_fold(X_train, y_train, best_params, epochs=30)

# 10. Evaluating the final tuned model on the test set
final_model.eval()
with torch.no_grad():
    y_pred = (final_model(torch.tensor(X_test, dtype=torch.float32)) > 0.5).int().numpy()
print("\nMLP Multi-output Model Results on Test Set:\n")
print(classification_report(y_test, y_pred, target_names=targets))

# 11. Saving the final trained model and creating folder if not present
os.makedirs("models/0pct", exist_ok=True)
torch.save(final_model.state_dict(), "models/0pct/mlp_multioutput.pth")


Params {'hidden1': 128, 'hidden2': 64, 'dropout': 0.3, 'lr': 0.001} -> CV weighted F1-score: 0.0000
Params {'hidden1': 256, 'hidden2': 128, 'dropout': 0.4, 'lr': 0.0005} -> CV weighted F1-score: 0.0000
Best hyperparameters selected: {'hidden1': 128, 'hidden2': 64, 'dropout': 0.3, 'lr': 0.001}

MLP Multi-output Model Results on Test Set:

              precision    recall  f1-score   support

     oestrus       0.00      0.00      0.00       241
     calving       0.00      0.00      0.00       137
    lameness       0.00      0.00      0.00       115
    mastitis       0.00      0.00      0.00        41

   micro avg       0.00      0.00      0.00       534
   macro avg       0.00      0.00      0.00       534
weighted avg       0.00      0.00      0.00       534
 samples avg       0.00      0.00      0.00       534



C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\vishn\anaconda3\Lib\site-packages\sklea

#### TABNET CLASSIFIER

In [11]:

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from pytorch_tabnet.tab_model import TabNetClassifier
import torch
import os

# 2. Loading my dataset (reduced enhanced dataset with no oversampling)
df = pd.read_csv("full_data_unhealthy_imputed_reduced_enhanced.csv")
targets = ['oestrus', 'calving', 'lameness', 'mastitis']
features = ['IN_ALLEYS', 'REST', 'EAT', 'ACTIVITY_LEVEL', 'hour_bin']

# 3. Performing feature engineering to add cyclical encoding of time and behavioral ratios
def create_features(X):
    X = X.copy()
    X['hour_sin'] = np.sin(2 * np.pi * X['hour_bin'] / 24)
    X['hour_cos'] = np.cos(2 * np.pi * X['hour_bin'] / 24)
    X['eat_rest_ratio'] = X['EAT'] / (X['REST'] + 1e-6)
    X['activity_rest_ratio'] = np.abs(X['ACTIVITY_LEVEL']) / (X['REST'] + 1e-6)
    return X.drop(columns=['hour_bin'])

X = create_features(df[features]).values
y = df[targets].values

# 4. Splitting my dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 5. Creating an output folder for saving models
os.makedirs("models/0pct", exist_ok=True)

# 6. Setting fixed hyperparameters for TabNet (chosen for faster run)
tabnet_params = {
    "n_d": 32,
    "n_a": 32,
    "n_steps": 5,
    "gamma": 1.5,
    "lambda_sparse": 1e-4
}

# 7. Training TabNet models for each target variable (no cross-validation, single run)
for target_idx, target in enumerate(targets):
    print(f"\n=== Training Fast TabNet for target: {target} ===")
    
    y_train_target = y_train[:, target_idx]
    y_test_target = y_test[:, target_idx]

    final_clf = TabNetClassifier(
        n_d=tabnet_params["n_d"],
        n_a=tabnet_params["n_a"],
        n_steps=tabnet_params["n_steps"],
        gamma=tabnet_params["gamma"],
        lambda_sparse=tabnet_params["lambda_sparse"],
        optimizer_fn=torch.optim.Adam,
        optimizer_params=dict(lr=1e-3),
        verbose=10,
        seed=42
    )

    final_clf.fit(
        X_train, y_train_target,
        eval_set=[(X_test, y_test_target)],
        eval_metric=['accuracy'],
        max_epochs=50,            # Reduced epochs
        patience=10,
        batch_size=2048,          # Larger batch for speed
        virtual_batch_size=256
    )

    preds_test = final_clf.predict(X_test)
    print(f"\nClassification report for Fast TabNet - {target}:\n")
    print(classification_report(y_test_target, preds_test))

    # Saving the trained TabNet model
    final_clf.save_model(f"models/0pct/tabnet_{target}_fast")



=== Training Fast TabNet for target: oestrus ===


C:\Users\vishn\anaconda3\Lib\site-packages\pytorch_tabnet\abstract_model.py:82: UserWarning: Device used : cpu
  warnings.warn(f"Device used : {self.device}")


epoch 0  | loss: 0.08643 | val_0_accuracy: 0.99587 |  0:00:31s
epoch 10 | loss: 0.02567 | val_0_accuracy: 0.99621 |  0:05:16s

Early stopping occurred at epoch 12 with best_epoch = 2 and best_val_0_accuracy = 0.99621


C:\Users\vishn\anaconda3\Lib\site-packages\pytorch_tabnet\callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)



Classification report for Fast TabNet - oestrus:

              precision    recall  f1-score   support

           0       1.00      1.00      1.00     63393
           1       0.00      0.00      0.00       241

    accuracy                           1.00     63634
   macro avg       0.50      0.50      0.50     63634
weighted avg       0.99      1.00      0.99     63634

Successfully saved model at models/0pct/tabnet_oestrus_fast.zip

=== Training Fast TabNet for target: calving ===


...

C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\vishn\anaconda3\Lib\site-packag

epoch 0  | loss: 0.05844 | val_0_accuracy: 0.99936 |  0:00:27s
epoch 10 | loss: 0.00449 | val_0_accuracy: 0.99936 |  0:05:01s

Early stopping occurred at epoch 10 with best_epoch = 0 and best_val_0_accuracy = 0.99936


C:\Users\vishn\anaconda3\Lib\site-packages\pytorch_tabnet\callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)



Classification report for Fast TabNet - mastitis:

              precision    recall  f1-score   support

           0       1.00      1.00      1.00     63593
           1       0.00      0.00      0.00        41

    accuracy                           1.00     63634
   macro avg       0.50      0.50      0.50     63634
weighted avg       1.00      1.00      1.00     63634

Successfully saved model at models/0pct/tabnet_mastitis_fast.zip


C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


#### LSTM MODEL

In [13]:

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import os

# 2. Loading my dataset (reduced enhanced dataset with no oversampling)
df = pd.read_csv("full_data_unhealthy_imputed_reduced_enhanced.csv")
targets = ['oestrus', 'calving', 'lameness', 'mastitis']
features = ['IN_ALLEYS', 'REST', 'EAT', 'ACTIVITY_LEVEL', 'hour_bin']

# 3. Creating sequences per cow-date
def create_features(X):
    X = X.copy()
    X['hour_sin'] = np.sin(2 * np.pi * X['hour_bin'] / 24)
    X['hour_cos'] = np.cos(2 * np.pi * X['hour_bin'] / 24)
    X['eat_rest_ratio'] = X['EAT'] / (X['REST'] + 1e-6)
    X['activity_rest_ratio'] = np.abs(X['ACTIVITY_LEVEL']) / (X['REST'] + 1e-6)
    return X.drop(columns=['hour_bin'])

# Apply feature engineering
df_features = create_features(df[features])
feature_cols = df_features.columns.tolist()

# Combine back engineered features into df
for col in feature_cols:
    df[col] = df_features[col]

# 4. Prepare sequences: group by cow and date
grouped = df.groupby(["cow", "date"])
X_seq = []
y_seq = {t: [] for t in targets}

for _, group in grouped:
    if len(group) == 3:  # keep only full sequences of 3 timesteps
        X_seq.append(group[feature_cols].values)
        for t in targets:
            y_seq[t].append(int(group[t].max()))  # label = 1 if any timestep has disease

X_seq = np.stack(X_seq)
y_seq = {t: np.array(v) for t, v in y_seq.items()}

print("Final sequence shape:", X_seq.shape)

# 5. Train/test split (same for all targets)
X_train, X_test = train_test_split(X_seq, test_size=0.2, random_state=42)

# 6. Dataset class for LSTM
class SeqDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# 7. LSTM model class
class LSTMModel(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, num_layers=2):
        super(LSTMModel, self).__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_dim, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        out, _ = self.lstm(x)
        out = out[:, -1, :]  # take output of last timestep
        out = self.fc(out)
        return self.sigmoid(out)

# 8. Function to train a single LSTM model
def train_lstm_model(X_train, y_train, X_test, y_test, target, epochs=10, batch_size=64):
    train_ds = SeqDataset(X_train, y_train)
    test_ds = SeqDataset(X_test, y_test)
    train_dl = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    test_dl = DataLoader(test_ds, batch_size=batch_size)

    model = LSTMModel(input_dim=X_train.shape[2])
    criterion = nn.BCELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

    for epoch in range(epochs):
        model.train()
        for xb, yb in train_dl:
            optimizer.zero_grad()
            preds = model(xb).squeeze()
            loss = criterion(preds, yb)
            loss.backward()
            optimizer.step()
        print(f"Epoch {epoch+1}/{epochs}, Loss: {loss.item():.4f}")

    # Evaluate
    model.eval()
    preds_all = []
    true_all = []
    with torch.no_grad():
        for xb, yb in test_dl:
            preds = model(xb).squeeze()
            preds_all.extend((preds > 0.5).cpu().numpy())
            true_all.extend(yb.cpu().numpy())

    print(f"\nClassification report for LSTM - {target}:\n")
    print(classification_report(true_all, preds_all))

    # Save model
    os.makedirs("models/0pct", exist_ok=True)
    torch.save(model.state_dict(), f"models/0pct/lstm_{target}.pt")

# 9. Train a model for each target
for target in targets:
    print(f"\n=== Training LSTM for target: {target} ===")
    y_train_target, y_test_target = train_test_split(y_seq[target], test_size=0.2, random_state=42)
    train_lstm_model(X_train, y_train_target, X_test, y_test_target, target)


Final sequence shape: (4434, 3, 8)

=== Training LSTM for target: oestrus ===
Epoch 1/10, Loss: 0.0078
Epoch 2/10, Loss: 0.1903
Epoch 3/10, Loss: 0.0084
Epoch 4/10, Loss: 0.0097
Epoch 5/10, Loss: 0.0134
Epoch 6/10, Loss: 0.0060
Epoch 7/10, Loss: 0.0048
Epoch 8/10, Loss: 0.0772
Epoch 9/10, Loss: 0.0249
Epoch 10/10, Loss: 0.0127

Classification report for LSTM - oestrus:

              precision    recall  f1-score   support

         0.0       0.99      1.00      0.99       877
         1.0       0.00      0.00      0.00        10

    accuracy                           0.99       887
   macro avg       0.49      0.50      0.50       887
weighted avg       0.98      0.99      0.98       887


=== Training LSTM for target: calving ===


C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Epoch 1/10, Loss: 0.0076
Epoch 2/10, Loss: 0.0033
Epoch 3/10, Loss: 0.0022
Epoch 4/10, Loss: 0.0019
Epoch 5/10, Loss: 0.0016
Epoch 6/10, Loss: 0.0015
Epoch 7/10, Loss: 0.0049
Epoch 8/10, Loss: 0.0013
Epoch 9/10, Loss: 0.0013
Epoch 10/10, Loss: 0.0012

Classification report for LSTM - calving:

              precision    recall  f1-score   support

         0.0       1.00      1.00      1.00       885
         1.0       0.00      0.00      0.00         2

    accuracy                           1.00       887
   macro avg       0.50      0.50      0.50       887
weighted avg       1.00      1.00      1.00       887


=== Training LSTM for target: lameness ===


C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Epoch 1/10, Loss: 0.0072
Epoch 2/10, Loss: 0.0031
Epoch 3/10, Loss: 0.0021
Epoch 4/10, Loss: 0.0015
Epoch 5/10, Loss: 0.0013
Epoch 6/10, Loss: 0.0012
Epoch 7/10, Loss: 0.0011
Epoch 8/10, Loss: 0.0010
Epoch 9/10, Loss: 0.0011
Epoch 10/10, Loss: 0.0009

Classification report for LSTM - lameness:

              precision    recall  f1-score   support

         0.0       1.00      1.00      1.00       886
         1.0       0.00      0.00      0.00         1

    accuracy                           1.00       887
   macro avg       0.50      0.50      0.50       887
weighted avg       1.00      1.00      1.00       887


=== Training LSTM for target: mastitis ===


C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Epoch 1/10, Loss: 0.0091
Epoch 2/10, Loss: 0.0037
Epoch 3/10, Loss: 0.0028
Epoch 4/10, Loss: 0.2256
Epoch 5/10, Loss: 0.0025
Epoch 6/10, Loss: 0.2322
Epoch 7/10, Loss: 0.0023
Epoch 8/10, Loss: 0.0022
Epoch 9/10, Loss: 0.2318
Epoch 10/10, Loss: 0.0024

Classification report for LSTM - mastitis:

              precision    recall  f1-score   support

         0.0       1.00      1.00      1.00       886
         1.0       0.00      0.00      0.00         1

    accuracy                           1.00       887
   macro avg       0.50      0.50      0.50       887
weighted avg       1.00      1.00      1.00       887



C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


#### TCN CLASSIFIER

In [15]:

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import classification_report
import numpy as np
import os

# 2. Reuse X_seq and y_seq from the LSTM step (shape: (4434, 3, 8))
# If not in memory, ensure to run the LSTM preparation code first

# 3. Train/test split
from sklearn.model_selection import train_test_split
X_train, X_test = train_test_split(X_seq, test_size=0.2, random_state=42)

# 4. Dataset class for TCN
class SeqDataset(Dataset):
    def __init__(self, X, y):
        # TCN expects (batch, channels, timesteps)
        X = np.transpose(X, (0, 2, 1))  # convert (batch, timesteps, features) -> (batch, features, timesteps)
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# 5. TCN block
class Chomp1d(nn.Module):
    def __init__(self, chomp_size):
        super(Chomp1d, self).__init__()
        self.chomp_size = chomp_size
    def forward(self, x):
        return x[:, :, :-self.chomp_size].contiguous()

class TemporalBlock(nn.Module):
    def __init__(self, n_inputs, n_outputs, kernel_size, stride, dilation, padding, dropout=0.2):
        super(TemporalBlock, self).__init__()
        self.conv1 = nn.Conv1d(n_inputs, n_outputs, kernel_size,
                               stride=stride, padding=padding, dilation=dilation)
        self.chomp1 = Chomp1d(padding)
        self.relu1 = nn.ReLU()
        self.dropout1 = nn.Dropout(dropout)

        self.conv2 = nn.Conv1d(n_outputs, n_outputs, kernel_size,
                               stride=stride, padding=padding, dilation=dilation)
        self.chomp2 = Chomp1d(padding)
        self.relu2 = nn.ReLU()
        self.dropout2 = nn.Dropout(dropout)

        self.net = nn.Sequential(self.conv1, self.chomp1, self.relu1, self.dropout1,
                                 self.conv2, self.chomp2, self.relu2, self.dropout2)
        self.downsample = nn.Conv1d(n_inputs, n_outputs, 1) if n_inputs != n_outputs else None
        self.relu = nn.ReLU()
        self.init_weights()

    def init_weights(self):
        self.conv1.weight.data.normal_(0, 0.01)
        self.conv2.weight.data.normal_(0, 0.01)
        if self.downsample is not None:
            self.downsample.weight.data.normal_(0, 0.01)

    def forward(self, x):
        out = self.net(x)
        res = x if self.downsample is None else self.downsample(x)
        return self.relu(out + res)

class TCNModel(nn.Module):
    def __init__(self, num_inputs, num_channels, kernel_size=2, dropout=0.2):
        super(TCNModel, self).__init__()
        layers = []
        num_levels = len(num_channels)
        for i in range(num_levels):
            dilation_size = 2 ** i
            in_channels = num_inputs if i == 0 else num_channels[i-1]
            out_channels = num_channels[i]
            layers += [TemporalBlock(in_channels, out_channels, kernel_size, stride=1,
                                     dilation=dilation_size, padding=(kernel_size-1)*dilation_size,
                                     dropout=dropout)]
        self.network = nn.Sequential(*layers)
        self.fc = nn.Linear(num_channels[-1], 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        # x: (batch, channels, timesteps)
        y = self.network(x)
        y = y[:, :, -1]  # last timestep
        y = self.fc(y)
        return self.sigmoid(y)

# 6. Function to train a single TCN model
def train_tcn_model(X_train, y_train, X_test, y_test, target, epochs=10, batch_size=64):
    train_ds = SeqDataset(X_train, y_train)
    test_ds = SeqDataset(X_test, y_test)
    train_dl = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    test_dl = DataLoader(test_ds, batch_size=batch_size)

    model = TCNModel(num_inputs=X_train.shape[2], num_channels=[32, 32, 32], kernel_size=2, dropout=0.1)
    criterion = nn.BCELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

    for epoch in range(epochs):
        model.train()
        for xb, yb in train_dl:
            optimizer.zero_grad()
            preds = model(xb).squeeze()
            loss = criterion(preds, yb)
            loss.backward()
            optimizer.step()
        print(f"Epoch {epoch+1}/{epochs}, Loss: {loss.item():.4f}")

    # Evaluate
    model.eval()
    preds_all = []
    true_all = []
    with torch.no_grad():
        for xb, yb in test_dl:
            preds = model(xb).squeeze()
            preds_all.extend((preds > 0.5).cpu().numpy())
            true_all.extend(yb.cpu().numpy())

    print(f"\nClassification report for TCN - {target}:\n")
    print(classification_report(true_all, preds_all))

    # Save model
    os.makedirs("models/0pct", exist_ok=True)
    torch.save(model.state_dict(), f"models/0pct/tcn_{target}.pt")

# 7. Train a TCN model for each target
for target in targets:
    print(f"\n=== Training TCN for target: {target} ===")
    y_train_target, y_test_target = train_test_split(y_seq[target], test_size=0.2, random_state=42)
    train_tcn_model(X_train, y_train_target, X_test, y_test_target, target)



=== Training TCN for target: oestrus ===
Epoch 1/10, Loss: 0.0000
Epoch 2/10, Loss: 0.0084
Epoch 3/10, Loss: 3.7130
Epoch 4/10, Loss: 0.0296
Epoch 5/10, Loss: 0.0044
Epoch 6/10, Loss: 0.2374
Epoch 7/10, Loss: 0.0207
Epoch 8/10, Loss: 0.0009
Epoch 9/10, Loss: 0.0009
Epoch 10/10, Loss: 0.0073

Classification report for TCN - oestrus:

              precision    recall  f1-score   support

         0.0       0.99      1.00      0.99       877
         1.0       0.00      0.00      0.00        10

    accuracy                           0.99       887
   macro avg       0.49      0.50      0.50       887
weighted avg       0.98      0.99      0.98       887


=== Training TCN for target: calving ===


C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Epoch 1/10, Loss: 0.0000
Epoch 2/10, Loss: 0.0000
Epoch 3/10, Loss: 0.0000
Epoch 4/10, Loss: 0.0000
Epoch 5/10, Loss: 0.0000
Epoch 6/10, Loss: 0.0000
Epoch 7/10, Loss: 0.0000
Epoch 8/10, Loss: 0.0000
Epoch 9/10, Loss: 0.0000
Epoch 10/10, Loss: 1.8356

Classification report for TCN - calving:

              precision    recall  f1-score   support

         0.0       1.00      1.00      1.00       885
         1.0       0.00      0.00      0.00         2

    accuracy                           1.00       887
   macro avg       0.50      0.50      0.50       887
weighted avg       1.00      1.00      1.00       887


=== Training TCN for target: lameness ===


C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Epoch 1/10, Loss: 0.0000
Epoch 2/10, Loss: 0.0000
Epoch 3/10, Loss: 0.0000
Epoch 4/10, Loss: 0.0000
Epoch 5/10, Loss: 0.0000
Epoch 6/10, Loss: 0.0000
Epoch 7/10, Loss: 0.0000
Epoch 8/10, Loss: 0.0000
Epoch 9/10, Loss: 0.0000
Epoch 10/10, Loss: 0.0000

Classification report for TCN - lameness:

              precision    recall  f1-score   support

         0.0       1.00      1.00      1.00       886
         1.0       0.00      0.00      0.00         1

    accuracy                           1.00       887
   macro avg       0.50      0.50      0.50       887
weighted avg       1.00      1.00      1.00       887


=== Training TCN for target: mastitis ===


C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Epoch 1/10, Loss: 0.0000
Epoch 2/10, Loss: 0.0000
Epoch 3/10, Loss: 0.0000
Epoch 4/10, Loss: 0.0000
Epoch 5/10, Loss: 0.0000
Epoch 6/10, Loss: 0.0000
Epoch 7/10, Loss: 0.0000
Epoch 8/10, Loss: 0.0000
Epoch 9/10, Loss: 0.0000
Epoch 10/10, Loss: 0.0000

Classification report for TCN - mastitis:

              precision    recall  f1-score   support

         0.0       1.00      1.00      1.00       886
         1.0       0.00      0.00      0.00         1

    accuracy                           1.00       887
   macro avg       0.50      0.50      0.50       887
weighted avg       1.00      1.00      1.00       887



C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


#### HYBRID ENSEMBLE METHOD

In [17]:

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, KFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
import joblib
import os


# (RF, LightGBM, TabNet, LSTM, TCN models already trained are taken here)

# Step 1: Prepare Dataset

df = pd.read_csv("full_data_unhealthy_imputed_reduced_enhanced.csv")
targets = ['oestrus', 'calving', 'lameness', 'mastitis']
features = ['IN_ALLEYS', 'REST', 'EAT', 'ACTIVITY_LEVEL', 'hour_bin']

def create_features(X):
    X = X.copy()
    X['hour_sin'] = np.sin(2 * np.pi * X['hour_bin']/24)
    X['hour_cos'] = np.cos(2 * np.pi * X['hour_bin']/24)
    X['eat_rest_ratio'] = X['EAT'] / (X['REST'] + 1e-6)
    X['activity_rest_ratio'] = np.abs(X['ACTIVITY_LEVEL']) / (X['REST'] + 1e-6)
    return X.drop(columns=['hour_bin'])

X = create_features(df[features])
y = df[targets]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

os.makedirs("models/ensemble", exist_ok=True)


# Step 2: Helper function for base model prediction

def get_model_predictions(model, X_train, y_train, X_test, model_type, target):
    """Generates OOF (out-of-fold) predictions for meta-features."""
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    oof_preds = np.zeros(len(X_train))
    test_preds = np.zeros(len(X_test))

    for train_idx, val_idx in kf.split(X_train):
        X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

        model.fit(X_tr, y_tr)
        oof_preds[val_idx] = model.predict_proba(X_val)[:, 1]
        test_preds += model.predict_proba(X_test)[:, 1] / 5

    return oof_preds, test_preds


# Step 3: Generate meta-features

meta_features_train = {}
meta_features_test = {}

from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier

# Base models
base_models = {
    "rf": RandomForestClassifier(n_estimators=150, max_depth=20, class_weight="balanced", random_state=42),
    "lgbm": LGBMClassifier(n_estimators=300, max_depth=16, learning_rate=0.1,
                            subsample=0.8, colsample_bytree=0.8, class_weight="balanced", random_state=42),
    # TabNet, LSTM, TCN can also be added here later
}

for target in targets:
    meta_train_target = []
    meta_test_target = []

    for model_name, model in base_models.items():
        print(f"Generating meta-features for {target} using {model_name}...")
        oof, preds = get_model_predictions(model, X_train, y_train[target], X_test, model_name, target)
        meta_train_target.append(oof)
        meta_test_target.append(preds)

    meta_features_train[target] = np.vstack(meta_train_target).T
    meta_features_test[target] = np.vstack(meta_test_target).T


# Step 4: Meta-learner training

final_predictions = {}

for target in targets:
    print(f"\n=== Training meta-learner for {target} ===")
    X_meta_train = meta_features_train[target]
    X_meta_test = meta_features_test[target]
    y_t = y_train[target]

    meta_model = LogisticRegression()
    meta_model.fit(X_meta_train, y_t)

    preds = meta_model.predict(X_meta_test)
    final_predictions[target] = preds

    print(f"\nClassification Report (Ensemble) - {target}:\n")
    print(classification_report(y_test[target], preds))

    # Save meta model
    joblib.dump(meta_model, f"models/ensemble/meta_{target}.pkl")

# Combine all predictions into a DataFrame
ensemble_preds_df = pd.DataFrame(final_predictions)
ensemble_preds_df.to_csv("models/ensemble/ensemble_predictions.csv", index=False)


Generating meta-features for oestrus using rf...
Generating meta-features for oestrus using lgbm...
[LightGBM] [Info] Number of positive: 821, number of negative: 202804
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001201 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1538
[LightGBM] [Info] Number of data points in the train set: 203625, number of used features: 8
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000
[LightGBM] [Info] Number of positive: 821, number of negative: 202804
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001086 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1538
[LightGBM] [Inf

C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


              precision    recall  f1-score   support

           0       1.00      1.00      1.00     63393
           1       0.00      0.00      0.00       241

    accuracy                           1.00     63634
   macro avg       0.50      0.50      0.50     63634
weighted avg       0.99      1.00      0.99     63634


=== Training meta-learner for calving ===

Classification Report (Ensemble) - calving:



C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\vishn\anaconda3\Lib\site-packag

              precision    recall  f1-score   support

           0       1.00      1.00      1.00     63497
           1       0.00      0.00      0.00       137

    accuracy                           1.00     63634
   macro avg       0.50      0.50      0.50     63634
weighted avg       1.00      1.00      1.00     63634


=== Training meta-learner for lameness ===

Classification Report (Ensemble) - lameness:

              precision    recall  f1-score   support

           0       1.00      1.00      1.00     63519
           1       0.00      0.00      0.00       115

    accuracy                           1.00     63634
   macro avg       0.50      0.50      0.50     63634
weighted avg       1.00      1.00      1.00     63634


=== Training meta-learner for mastitis ===


C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\vishn\anaconda3\Lib\site-packag


Classification Report (Ensemble) - mastitis:

              precision    recall  f1-score   support

           0       1.00      1.00      1.00     63593
           1       0.00      0.00      0.00        41

    accuracy                           1.00     63634
   macro avg       0.50      0.50      0.50     63634
weighted avg       1.00      1.00      1.00     63634

